In [1]:
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import cv2

In [ ]:
DATA_PATH_SYNCED = "./kitti_raw_sync"
DATA_PATH_UNSYNCED = "./kitti_raw_unsynced"

OUTPUT_PATH = "./mulitview_kitti"
MAPPING_PATH = "./mapping"
TRAIN_RAND_FILE = "train_rand.txt"
TRAIN_RAND_PATH = os.path.join(MAPPING_PATH, TRAIN_RAND_FILE)
TRAIN_MAPPING_FILE = "train_mapping.txt"
TRAIN_MAPPING_PATH = os.path.join(MAPPING_PATH, TRAIN_MAPPING_FILE)

### Mapping from Detection benchmark files to raw data files

In [3]:
# read train_rand.txt
with open(TRAIN_RAND_PATH, 'r') as f:
    train_rand = f.readlines()
train_rand = train_rand[0].split(",")
train_rand = np.array(train_rand, dtype=np.int32)

# read train_mapping.txt
with open(TRAIN_MAPPING_PATH, 'r') as f:
    train_mapping = f.readlines()

In [4]:
def find_idx_clostes_timestamp(timestamp0, timestamps_path):
    with open(timestamps_path, 'r') as f:
        timestamps = f.readlines()
    # make a list of datetime64 objects timestamps1 list
    timestamps = [np.datetime64(f"{timestamp.split(' ')[0]}T{timestamp.split(' ')[1]}") for timestamp in timestamps]
    # find index to the closest timestamp to t1
    idx = np.argmin(np.abs(np.array(timestamps) - timestamp0))
    frame_num = str(idx).zfill(10)
    return frame_num, timestamps[idx]

In [5]:
def get_camera_params(calib_cam):
    for l in calib_cam:
        l = l.rstrip().split(" ")
        if l[0] == "D_00:":
            D_00 = np.array(l[1:]).astype(np.float32)
        if l[0] == "D_01:":
            D_01 = np.array(l[1:]).astype(np.float32)
        if l[0] == "D_02:":
            D_02 = np.array(l[1:]).astype(np.float32)
        if l[0] == "D_03:":
            D_03 = np.array(l[1:]).astype(np.float32)
        if l[0] == "T_00:":
            T_00 = np.array(l[1:]).astype(np.float32)
        if l[0] == "T_01:":
            T_01 = np.array(l[1:]).astype(np.float32)
        if l[0] == "T_02:":
            T_02 = np.array(l[1:]).astype(np.float32)
        if l[0] == "T_03:":
            T_03 = np.array(l[1:]).astype(np.float32)
        if l[0] == "R_00:":
            R_00 = np.array(l[1:]).astype(np.float32)
        if l[0] == "R_01:":    
            R_01 = np.array(l[1:]).astype(np.float32)
        if l[0] == "R_02:":
            R_02 = np.array(l[1:]).astype(np.float32)
        if l[0] == "R_03:":    
            R_03 = np.array(l[1:]).astype(np.float32)
        if l[0] == "K_00:":
            K_00 = np.array(l[1:]).astype(np.float32)
        if l[0] == "K_01:":
            K_01 = np.array(l[1:]).astype(np.float32)
        if l[0] == "K_02:":
            K_02 = np.array(l[1:]).astype(np.float32)
        if l[0] == "K_03:":
            K_03 = np.array(l[1:]).astype(np.float32)
        if l[0] == "R_rect_00:":
            R_rect_00 = np.array(l[1:]).astype(np.float32)
    return D_00, D_01, D_02, D_03, T_00, T_01, T_02, T_03, R_00, R_01, R_02, R_03, K_00, K_01, K_02, K_03, R_rect_00

In [7]:
for i, num in tqdm(enumerate(train_rand)):
    # idx for object detection
    object_detection_idx = str(i).zfill(6)
    # find sequence in raw data
    seq = train_mapping[num-1]
    seq = seq.split(" ")
    seq_dir_name = seq[0]
    seq_drive_dir_name = seq[1]

    frame_num = seq[2]
    # remove newline character
    frame_num = frame_num[:-1]

    # search for timestamp for cam0 in synced data
    timestamp0_path = os.path.join(DATA_PATH_SYNCED, seq_dir_name, seq_drive_dir_name, "image_00", "timestamps.txt")

    with open(timestamp0_path, 'r') as f:
        timestamps0 = f.readlines()
    idx = int(frame_num)
    timestamp0 = timestamps0[idx]
    date0 = timestamp0.split(" ")[0]
    times0 = timestamp0.split(" ")[1]

    datetime_str_ns = f"{date0}T{times0}"
    t0_sync = np.datetime64(datetime_str_ns)
    
    # now we have the timestamp for cam0 and we need to find the corresponding timestamp in the unsynced data
    seq_drive_dir_name = seq_drive_dir_name.replace("sync", "extract")
    timestamp0_path_unsynced = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, seq_drive_dir_name, "image_00", "timestamps.txt")

    frame_num0, t0_unsynced = find_idx_clostes_timestamp(t0_sync, timestamp0_path_unsynced)

    if t0_sync != t0_unsynced:
        print("Error: timestamps are not in sync", idx, " ", num)

    cam0_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, seq_drive_dir_name, "image_00", "data", frame_num0 + ".png")

    # timestamp0 is the timestamp for cam0 and is synced with lidar and now we need to find the corresponding timestamp for cam1, cam2, cam3
    timestamp1_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, seq_drive_dir_name, "image_01", "timestamps.txt")
    frame_num1, t1 = find_idx_clostes_timestamp(t0_unsynced, timestamp1_path)
    cam1_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, seq_drive_dir_name, "image_01", "data", frame_num1 + ".png")

    # cam2
    timestamp2_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, seq_drive_dir_name, "image_02", "timestamps.txt")
    frame_num2, t2 = find_idx_clostes_timestamp(t0_unsynced, timestamp2_path)
    cam2_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, seq_drive_dir_name, "image_02", "data", frame_num2 + ".png")

    # cam3
    timestamp3_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, seq_drive_dir_name, "image_03", "timestamps.txt")
    frame_num3, t3 = find_idx_clostes_timestamp(t0_unsynced, timestamp3_path)
    cam3_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, seq_drive_dir_name, "image_03", "data", frame_num3 + ".png")
    
    # check if timestamps are in sync
    if t0_unsynced - t1 > np.timedelta64(15, 'ms'):
        print("Error: timestamps are not in sync", idx, " ", num)
    if t0_unsynced - t2 > np.timedelta64(15, 'ms'):
        print("Error: timestamps are not in sync", idx, " ", num)
    if t0_unsynced - t3 > np.timedelta64(15, 'ms'):
        print("Error: timestamps are not in sync", idx, " ", num)

    # get calibration data
    calib_cam_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, "calib_cam_to_cam.txt")
    calib_velo_cam_path = os.path.join(DATA_PATH_UNSYNCED, seq_dir_name, "calib_velo_to_cam.txt")

    with open(calib_cam_path, 'r') as f:
        calib_cam = f.readlines()
    # get calib params
    D_00, D_01, D_02, D_03, T_00, T_01, T_02, T_03, R_00, R_01, R_02, R_03, K_00, K_01, K_02, K_03, R_rect_00 = get_camera_params(calib_cam)
    # get lidar cam params
    with open(calib_velo_cam_path, 'r') as f:
        calib_velo_cam = f.readlines()
    for l in calib_velo_cam:
        l = l.rstrip().split(" ")
        if l[0] == "R:":
            R = np.array(l[1:]).astype(np.float32)
        if l[0] == "T:":
            T = np.array(l[1:]).astype(np.float32)
    
    # create output directory
    calib_o_path = os.path.join(OUTPUT_PATH, "data_object_calib", "training", "calib") 
    os.makedirs(calib_o_path, exist_ok=True)
    img0_o_path = os.path.join(OUTPUT_PATH, "data_object_image_0", "training", "image_0")
    os.makedirs(img0_o_path, exist_ok=True)
    img1_o_path = os.path.join(OUTPUT_PATH, "data_object_image_1", "training", "image_1")
    os.makedirs(img1_o_path, exist_ok=True)
    img2_o_path = os.path.join(OUTPUT_PATH, "data_object_image_2", "training", "image_2")
    os.makedirs(img2_o_path, exist_ok=True)
    img3_o_path = os.path.join(OUTPUT_PATH, "data_object_image_3", "training", "image_3")
    os.makedirs(img3_o_path, exist_ok=True)

    
    # read images
    img0 = plt.imread(cam0_path)
    img1 = plt.imread(cam1_path)
    img2 = plt.imread(cam2_path)[:,:,:3]
    img3 = plt.imread(cam3_path)[:,:,:3]


    h0, w0 = img0.shape[:2]
    K_00_new, roi_0 = cv2.getOptimalNewCameraMatrix(K_00.reshape((3,3)), D_00, (w0, h0), 1., (w0, h0))
    map1_0, map2_0 = cv2.initUndistortRectifyMap(K_00.reshape((3,3)), D_00, np.eye(3,3), K_00_new, (w0, h0), cv2.CV_16SC2)
    img0_undistorted = cv2.remap(img0, map1_0, map2_0, interpolation=cv2.INTER_CUBIC)
    x0, y0, w0n, h0n = roi_0
    img0_undistorted = img0_undistorted[y0:y0+h0n, x0:x0+w0n]
    K_00_new[0,2] = K_00_new[0,2] - x0
    K_00_new[1,2] = K_00_new[1,2] - y0

    h1, w1 = img1.shape[:2]
    K_01_new, roi_1 = cv2.getOptimalNewCameraMatrix(K_01.reshape((3,3)), D_01, (w1, h1), 1., (w1, h1))
    map1_1, map2_1 = cv2.initUndistortRectifyMap(K_01.reshape((3,3)), D_01, np.eye(3,3), K_01_new, (w1, h1), cv2.CV_16SC2)
    img1_undistorted = cv2.remap(img1, map1_1, map2_1, interpolation=cv2.INTER_CUBIC)
    x1, y1, w1n, h1n = roi_1
    img1_undistorted = img1_undistorted[y1:y1+h1n, x1:x1+w1n]
    K_01_new[0,2] = K_01_new[0,2] - x1
    K_01_new[1,2] = K_01_new[1,2] - y1

    h2, w2 = img2.shape[:2]
    K_02_new, roi_2 = cv2.getOptimalNewCameraMatrix(K_02.reshape((3,3)), D_02, (w2, h2), 1., (w2, h2))
    map1_2, map2_2 = cv2.initUndistortRectifyMap(K_02.reshape((3,3)), D_02, np.eye(3,3), K_02_new, (w2, h2), cv2.CV_16SC2)
    img2_undistorted = cv2.remap(img2, map1_2, map2_2, interpolation=cv2.INTER_CUBIC)
    x2, y2, w2n, h2n = roi_2
    img2_undistorted = img2_undistorted[y2:y2+h2n, x2:x2+w2n]
    K_02_new[0,2] = K_02_new[0,2] - x2
    K_02_new[1,2] = K_02_new[1,2] - y2

    h3, w3 = img3.shape[:2]
    K_03_new, roi_3 = cv2.getOptimalNewCameraMatrix(K_03.reshape((3,3)), D_03, (w3, h3), 1., (w3, h3))
    map1_3, map2_3 = cv2.initUndistortRectifyMap(K_03.reshape((3,3)), D_03, np.eye(3,3), K_03_new, (w3, h3), cv2.CV_16SC2)
    img3_undistorted = cv2.remap(img3, map1_3, map2_3, interpolation=cv2.INTER_CUBIC)
    x3, y3, w3n, h3n = roi_3
    img3_undistorted = img3_undistorted[y3:y3+h3n, x3:x3+w3n]
    K_03_new[0,2] = K_03_new[0,2] - x3
    K_03_new[1,2] = K_03_new[1,2] - y3


    plt.imsave(os.path.join(img0_o_path, object_detection_idx + ".png"), img0_undistorted, cmap='gray')
    plt.imsave(os.path.join(img1_o_path, object_detection_idx + ".png"), img1_undistorted, cmap='gray')
    plt.imsave(os.path.join(img2_o_path, object_detection_idx + ".png"), img2_undistorted)
    plt.imsave(os.path.join(img3_o_path, object_detection_idx + ".png"), img3_undistorted)

    # save calibration data
    calib_o_file = os.path.join(calib_o_path, object_detection_idx + ".txt")
    # write K_00, K_01, K_02, K_03, D_00, D_01, D_02, D_03, R_00, R_01, R_02, R_03, T_00, T_01, T_02, T_03, R0_rect to txt file
    with open(calib_o_file, 'w') as f:
        f.write("K0: " + " ".join(K_00_new.flatten().astype(str)) + "\n")
        f.write("K1: " + " ".join(K_01_new.flatten().astype(str)) + "\n")
        f.write("K2: " + " ".join(K_02_new.flatten().astype(str)) + "\n")
        f.write("K3: " + " ".join(K_03_new.flatten().astype(str)) + "\n")
        f.write("R0: " + " ".join(R_00.astype(str)) + "\n")
        f.write("R1: " + " ".join(R_01.astype(str)) + "\n")
        f.write("R2: " + " ".join(R_02.astype(str)) + "\n")
        f.write("R3: " + " ".join(R_03.astype(str)) + "\n")
        f.write("T0: " + " ".join(T_00.astype(str)) + "\n")
        f.write("T1: " + " ".join(T_01.astype(str)) + "\n")
        f.write("T2: " + " ".join(T_02.astype(str)) + "\n")
        f.write("T3: " + " ".join(T_03.astype(str)) + "\n")
        f.write("R0_rect: " + " ".join(R_rect_00.astype(str)) + "\n")
        f.write("T: " + " ".join(T.astype(str)) + "\n")
        f.write("R: " + " ".join(R.astype(str)) + "\n")

0it [00:00, ?it/s]/tmp/ipykernel_2056950/3221706788.py:25: DeprecationWarning: parsing timezone aware datetimes is deprecated; this will raise an error in the future
  t0_sync = np.datetime64(datetime_str_ns)
/tmp/ipykernel_2056950/4138176862.py:5: DeprecationWarning: parsing timezone aware datetimes is deprecated; this will raise an error in the future
  timestamps = [np.datetime64(f"{timestamp.split(' ')[0]}T{timestamp.split(' ')[1]}") for timestamp in timestamps]
7481it [19:07,  6.52it/s]
